## Trims audio from .data annotation

In [1]:
# ============================================================================
#  T R I M   S E G M E N T S   F R O M   A V I A N Z   . D A T A
# ============================================================================
#  Walk a root tree for *.wav.data annotation files, match each to its sibling
#  .wav, and emit time-cropped WAVs under a per-mode output folder.
#
#  Modes:
#    trim_to_annotation           crop each .data segment
#                                  -> {stem}_{n}_trimmed.wav
#    trim_to_interval             ignore annotations, slice whole wav into
#                                  fixed-length windows
#                                  -> {stem}_{n}_trimmed.wav
#    trim_annotation_to_interval  crop each segment, subdivide into windows
#                                  -> {stem}_{n}_{k}_trimmed.wav
#
#  Output folder (per source dir), one per mode:
#    trimmed_to_annotation/  trimmed_to_interval/  trimmed_annotation_to_interval/
#  Frequency bounds in each segment row are ignored (time-only crop).
# ============================================================================

import json
import logging
from pathlib import Path

import soundfile as sf

# ---- verbose logging (toggle DEBUG_VERBOSE to silence) ---------------------
DEBUG_VERBOSE = True
logging.basicConfig(level=logging.DEBUG if DEBUG_VERBOSE else logging.INFO,
                    format="%(levelname)s %(message)s")
log = logging.getLogger(__name__)

# ---- mode -> output folder name --------------------------------------------
_MODE_DIRS = {
    "trim_to_annotation": "trimmed_to_annotation",
    "trim_to_interval": "trimmed_to_interval",
    "trim_annotation_to_interval": "trimmed_annotation_to_interval",
}


def parse_data_segments(data_path):
    """Return list of (start_s, end_s) segment time bounds from an AviaNZ .data file.

    Reads the JSON array, discards index 0 (header dict: Operator/Reviewer/
    Duration/noiseLevel/noiseTypes), and extracts time bounds from each
    remaining segment row of form [start_s, end_s, low_hz, high_hz, [labels]].
    """
    with open(data_path, "r", encoding="utf-8") as fh:
        payload = json.load(fh)                 # full annotation array

    # index 0 is the metadata header dict; segments are the remaining rows
    segments = []                               # accumulate (start, end) pairs
    for row in payload[1:]:                      # skip header at index 0
        start_s = float(row[0])                 # row[0] = segment start (seconds)
        end_s = float(row[1])                   # row[1] = segment end   (seconds)
        # row[2], row[3] = low_hz, high_hz -> intentionally unused (time-only)
        segments.append((start_s, end_s))
    return segments


def _interval_windows(start_frame, end_frame, step, drop_remainder):
    """Return list of (w_start, w_end) fixed-length frame windows spanning a range.

    Tiles [start_frame, end_frame) into windows of 'step' frames. A trailing
    window shorter than 'step' is kept unless drop_remainder is True.
    """
    windows = []                                # accumulate sub-windows
    cursor = start_frame                        # sliding window start
    while cursor < end_frame:
        w_end = min(cursor + step, end_frame)   # clamp final window to range end
        if (w_end - cursor) < step and drop_remainder:
            break                               # discard trailing short remainder
        windows.append((cursor, w_end))
        cursor += step                          # advance by one window length
    return windows


def trim_one_wav(wav_path, data_path, mode="trim_to_annotation",
                 interval_s=None, drop_remainder=False):
    """Crop a single wav into trimmed clips under a per-mode output folder.

    mode selects the trimming strategy: 'trim_to_annotation' crops each .data
    segment; 'trim_to_interval' ignores annotations and tiles the whole wav into
    interval_s windows; 'trim_annotation_to_interval' crops each segment then
    tiles it into interval_s windows. interval_s (seconds) is required for the
    two interval modes. drop_remainder discards a trailing window shorter than
    interval_s. Cropping is frame-accurate via sample-rate frame conversion.
    """
    if mode not in _MODE_DIRS:                  # FALLBACK-GUARD: unknown mode
        raise ValueError(f"unknown mode: {mode!r}")
    if mode in ("trim_to_interval", "trim_annotation_to_interval") and not interval_s:
        raise ValueError(f"interval_s required for mode {mode!r}")

    info = sf.info(str(wav_path))               # probe sr / frame count w/o full read
    sr = info.samplerate                        # native sample rate (Hz)
    n_frames_total = info.frames                # total frames for clamping

    stem = wav_path.stem                        # filename without .wav extension
    out_dir = wav_path.parent / _MODE_DIRS[mode]    # per-mode subfolder beside wav
    out_dir.mkdir(parents=True, exist_ok=True)  # idempotent across repeated runs

    step = int(round(interval_s * sr)) if interval_s else None  # window len (frames)

    # ---- build the work list of (n, k, w_start, w_end) clip windows --------
    # n indexes the source unit (segment, or whole-file for trim_to_interval);
    # k indexes sub-windows within unit n (None when not subdividing).
    jobs = []                                   # accumulate clip specifications

    if mode == "trim_to_interval":
        # whole-file tiling: single unit n=0, k enumerates windows across file
        for k, (w0, w1) in enumerate(
                _interval_windows(0, n_frames_total, step, drop_remainder)):
            jobs.append((k, None, w0, w1))      # n-slot reused as window index
    else:
        # annotation-driven: iterate segments from the .data file
        segments = parse_data_segments(data_path)
        log.debug("  %d segment(s) in %s", len(segments), data_path.name)
        for n, (start_s, end_s) in enumerate(segments):     # n preserves .data order
            sf0 = max(0, int(round(start_s * sr)))          # segment start frame
            sf1 = min(n_frames_total, int(round(end_s * sr)))   # segment end frame
            if sf1 <= sf0:                      # FALLBACK-GUARD: empty/inverted span
                log.warning("  skip seg %d (empty span %.3f..%.3f s)",
                            n, start_s, end_s)
                continue

            if mode == "trim_to_annotation":
                jobs.append((n, None, sf0, sf1))            # whole segment, no k
            else:  # trim_annotation_to_interval: subdivide this segment
                for k, (w0, w1) in enumerate(
                        _interval_windows(sf0, sf1, step, drop_remainder)):
                    jobs.append((n, k, w0, w1))             # segment n, sub-window k

    # ---- write each clip ----------------------------------------------------
    for n, k, w_start, w_end in jobs:
        block, _ = sf.read(str(wav_path),
                           start=w_start,
                           stop=w_end,
                           dtype="float32")     # float32 keeps full dynamic range

        # 2-index name only when a sub-window index k is present
        if k is None:
            out_name = f"{stem}_{n}_trimmed.wav"
        else:
            out_name = f"{stem}_{n}_{k}_trimmed.wav"
        out_path = out_dir / out_name           # per-mode subfolder
        sf.write(str(out_path), block, sr,
                 subtype=info.subtype)          # mirror source bit-depth/subtype
        log.debug("  wrote %s (%d frames)", out_path.name, w_end - w_start)


def trim_tree(root_dir, mode="trim_to_annotation",
              interval_s=None, drop_remainder=False):
    """Recursively trim all wavs that have a sibling .wav.data under root_dir.

    Walks root_dir for *.wav.data files, pairs each with its sibling .wav, and
    trims in place under the per-mode output folder, preserving the existing
    directory tree. mode / interval_s / drop_remainder pass through to
    trim_one_wav. .data files lacking a sibling wav are skipped.
    """
    root = Path(root_dir)                        # tree root to search
    data_files = sorted(root.rglob("*.wav.data"))   # every annotation file in tree
    log.info("found %d .wav.data file(s) under %s [mode=%s]",
             len(data_files), root, mode)

    for data_path in data_files:
        # ".wav.data" -> sibling ".wav": strip the trailing ".data" suffix
        wav_path = data_path.with_suffix("")     # drops ".data", leaves ".wav"
        if not wav_path.is_file():               # FALLBACK-GUARD: orphan annotation
            log.warning("no sibling wav for %s -- skipped", data_path.name)
            continue
        log.info("trimming %s", wav_path.relative_to(root))
        trim_one_wav(wav_path, data_path, mode=mode,
                     interval_s=interval_s, drop_remainder=drop_remainder)


# ============================================================================
#  U S A G I
# ============================================================================
# from trim_segments import trim_tree
#
# # crop each annotation segment:
# trim_tree("path/to/root", mode="trim_to_annotation")
#
# # tile whole wav into 5 s windows (annotations ignored):
# trim_tree("path/to/root", mode="trim_to_interval", interval_s=5)
#
# # crop segments, then subdivide each into 3 s windows, drop short tails:
# trim_tree("path/to/root", mode="trim_annotation_to_interval",
#           interval_s=3, drop_remainder=True)


# Trim audio files to specified interval lengths, or annotations, and save in subdir next to original audio
AUDIO_DIR = r"test_audio"

trim_tree(AUDIO_DIR, mode="trim_to_interval", interval_s=10)

INFO found 0 .wav.data file(s) under test_audio [mode=trim_to_interval]
